# Аналитическая часть задания


## Анализ временных рядов

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("temperature_data.csv", parse_dates=["timestamp"])

Вычисление скользящего среднего и стандартного отклонения для сглаживания температурных колебаний. Нахождение аномалий. 

In [2]:
def analyzing_temp(df):

    df = df.sort_values(by=["city", "timestamp"])
    df["temp_rolling"] = df.groupby("city")["temperature"].transform(lambda x: x.rolling(window=30, min_periods=1).mean())  # Скользящее среднее
    seasons_data = (df.groupby(["city", "season"])["temperature"].agg(["mean", "std"]).reset_index().rename(columns={"mean": "season_mean", "std": "season_std"}))  # Среднее и стандартное отклонения
    df = df.merge(seasons_data, on=["city", "season"], how="left")
    df["anomaly"] = (df["temperature"] > df["season_mean"] + 2 * df["season_std"]) | (df["temperature"] < df["season_mean"] - 2 * df["season_std"])
    anomaly = df[df["anomaly"]]

    return df, seasons_data, anomaly


df, stats, anomalies = analyzing_temp(df)
print(stats.head())
print(anomalies.head())

      city  season  season_mean  season_std
0  Beijing  autumn    15.714550    5.169188
1  Beijing  spring    13.033795    5.117107
2  Beijing  summer    26.984436    5.228256
3  Beijing  winter    -2.053221    5.092932
4   Berlin  autumn    11.064073    4.965599
        city  timestamp  temperature  season  temp_rolling  season_mean  \
51   Beijing 2010-02-21   -14.587482  winter     -2.123867    -2.053221   
89   Beijing 2010-03-31     2.167881  spring     14.156990    13.033795   
126  Beijing 2010-05-07    -1.304205  spring     13.197526    13.033795   
146  Beijing 2010-05-27     2.785955  spring     12.168559    13.033795   
172  Beijing 2010-06-22    43.997862  summer     23.564699    26.984436   

     season_std  anomaly  
51     5.092932     True  
89     5.117107     True  
126    5.117107     True  
146    5.117107     True  
172    5.228256     True  


Создаем функцию для аналитики каждого города с распараллеливанием. У меня была проблема с компиляцией, поэтому функция анализа каждого города в отдельном файле "analyze_city.py". 

In [ ]:
from multiprocessing import Pool, cpu_count
from analyze_city import *

def analyzing_temp_parallel(df, n_workers=None):
    if n_workers is None:
        n_workers = min(cpu_count(), df["city"].nunique())

    city_groups = []
    for _, group in df.groupby("city"):
        city_groups.append(group)

    with Pool(n_workers) as pool:
        results = pool.map(analyze_city, city_groups)

    dfs = []
    seasons = []
    anomalies = []

    for result in results:
        df_part = result[0]
        season_part = result[1]
        anomaly_part = result[2]

        dfs.append(df_part)
        seasons.append(season_part)
        anomalies.append(anomaly_part)

    all_df = pd.concat(dfs, ignore_index=True)
    all_seasons = pd.concat(seasons, ignore_index=True)
    all_anomalies = pd.concat(anomalies, ignore_index=True)

    return all_df, all_seasons, all_anomalies


df = pd.read_csv("temperature_data.csv", parse_dates=["timestamp"])
df, stats, anomalies = analyzing_temp_parallel(df)

print(stats.head())
print(anomalies.head())

   season  season_mean  season_std     city
0  autumn    15.714550    5.169188  Beijing
1  spring    13.033795    5.117107  Beijing
2  summer    26.984436    5.228256  Beijing
3  winter    -2.053221    5.092932  Beijing
4  autumn    11.064073    4.965599   Berlin
      city  timestamp  temperature  season  temp_rolling  season_mean  \
0  Beijing 2010-02-21   -14.587482  winter     -2.123867    -2.053221   
1  Beijing 2010-03-31     2.167881  spring     14.156990    13.033795   
2  Beijing 2010-05-07    -1.304205  spring     13.197526    13.033795   
3  Beijing 2010-05-27     2.785955  spring     12.168559    13.033795   
4  Beijing 2010-06-22    43.997862  summer     23.564699    26.984436   

   season_std  anomaly  
0    5.092932     True  
1    5.117107     True  
2    5.117107     True  
3    5.117107     True  
4    5.228256     True  


Теперь сравним, что быстрее? Без распараллеливания или с ним. 

In [11]:
import time

def benchmark_functions(df, n_runs=3):
    seq_times = []
    par_times = []
    for _ in range(n_runs):
        df_copy = df.copy()

        start = time.time()
        analyzing_temp(df_copy)
        end = time.time()
        seq_times.append(end - start)

    for _ in range(n_runs):
        df_copy = df.copy()

        start = time.time()
        analyzing_temp_parallel(df_copy)
        end = time.time()
        par_times.append(end - start)

    seq_avg = sum(seq_times) / len(seq_times)
    par_avg = sum(par_times) / len(par_times)

    print("Обычная функция:", end="\n\n")
    print("Время выполнения за 1 раз:", seq_times, end="\n\n")
    print("Среднее время выполнения за несколько запросов:", seq_avg, end="\n\n")

    print("Распараллеленная функция:", end="\n\n")
    print("Время выполнения за 1 раз:", par_times, end="\n\n")
    print("Среднее время выполнения за несколько запросов:", par_avg, end="\n\n")

    speedup = seq_avg / par_avg
    if speedup > 1:
        print(f"Параллельная функция быстрее в {speedup} раз", end="\n\n")
    else:
        print(f"Параллельная функция медленнее в {1/speedup} раз", end="\n\n")

df = pd.read_csv("temperature_data.csv", parse_dates=["timestamp"])

benchmark_functions(df, n_runs=100)

Обычная функция:

Время выполнения за 1 раз: [0.0326390266418457, 0.027456045150756836, 0.026890993118286133, 0.026181936264038086, 0.02671194076538086, 0.02675485610961914, 0.027025938034057617, 0.026695728302001953, 0.026760101318359375, 0.02712082862854004, 0.02657008171081543, 0.026917219161987305, 0.026310205459594727, 0.026756763458251953, 0.026259899139404297, 0.026543140411376953, 0.027945995330810547, 0.028461933135986328, 0.030964136123657227, 0.030154943466186523, 0.034564971923828125, 0.03169679641723633, 0.029299259185791016, 0.028448104858398438, 0.028387069702148438, 0.0287168025970459, 0.029452085494995117, 0.028609037399291992, 0.028243303298950195, 0.029216766357421875, 0.028616905212402344, 0.028717756271362305, 0.027663230895996094, 0.028748750686645508, 0.028516054153442383, 0.02863001823425293, 0.02898883819580078, 0.027408123016357422, 0.02742314338684082, 0.026914119720458984, 0.02672100067138672, 0.02663397789001465, 0.026980876922607422, 0.026333332061767578, 

Я думаю, что распараллеленная функция оказалась настолько медленней, потому что мой объем данных недостаточно велик, и поэтому тут выгоднее использовать обычную версию, но в случае реальных и огромных массивов данных, я думаю, лучше себя покажет именно параллельная функция. Потому что, когда у нас обычная последовательная функция, то данные сразу выполняются, а распараллеленной нужна подготовка в виде выделения потоков и прочего, на маленьком датасете проще сразу начать анализировать, а на большом уже так будет не оптимально. 